# einops-repeat — ex6: causal attention mask — broadcast (T,T) → (B,H,T,T) and visualize

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-repeat`. Running the final beacon cell reports progress against the `Einops: Repeat` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Repeat` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einops-repeat`** (exercise 6). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-repeat"
DD_SUBTOPIC = "Einops: Repeat"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## einops.repeat — quick refresher

`repeat(tensor, pattern, **axes_lengths)` introduces new axes or stretches existing ones:
1. **New axis** — `'h w -> b h w'` with `b=4` broadcasts across a new leading dim.
2. **Stretch (nearest-neighbor)** — `'h w -> (h r) w'` with `r=2` makes each row appear twice in a contiguous block (rows 0,0,1,1,2,2,...).
3. **Tile** — `'h w -> h (r w)'` with `r=2` concatenates two full copies side-by-side (cols 0..w-1, then 0..w-1 again).

Stretch vs tile: in the composite `(a b)` the axis written **first varies slower**. `(h r)` puts source row 0 at output rows `0..r-1`; `(r h)` puts source row 0 at output rows `0, h, 2h, ...`. The new exercises lean on this distinction repeatedly.

### Exercise 6 — causal attention mask — broadcast (T,T) → (B,H,T,T) and visualize

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Use repeat to lift a 2D causal mask into the (batch, head, query, key) shape Transformer attention expects, and visualize it as a heatmap.
> Keywords: attention, mask, broadcast, visualization
> ```

**KCs targeted:** `repeat-add-axis`, `repeat-multi-axis-broadcast`

Implement `ex6_broadcast_causal_mask(mask_2d, b, h)`.

Input: `mask_2d` is a `(T, T)` float tensor (0.0 = keep, -inf = block) — a lower-triangular causal mask used inside one attention head. Output: `(b, h, T, T)`, where every (batch, head) slice is the **same** mask broadcast in.

Use a **single** `einops.repeat` call with two new named axes. Do **not** use `unsqueeze`, `expand`, `tile`, or `torch.stack`.

After your function passes its shape/value asserts, the test cell also plots `mask_2d` as a matplotlib heatmap so you can see the staircase pattern of allowed (white) vs blocked (dark) query→key positions. The plot is for you; the asserts are what grade the answer.

In [ ]:
def ex6_broadcast_causal_mask(mask_2d: Tensor, b: int, h: int) -> Tensor:
    """Broadcast a (T, T) causal mask up to (b, h, T, T) with a single einops.repeat."""
    raise NotImplementedError()


def _test_ex6():
    T = 6
    # Build a standard causal mask: 0.0 on-and-below diagonal, -inf above.
    mask_2d = t.zeros(T, T)
    mask_2d.masked_fill_(t.triu(t.ones(T, T, dtype=t.bool), diagonal=1), float('-inf'))

    b, h = 2, 4
    out = ex6_broadcast_causal_mask(mask_2d, b=b, h=h)

    assert out.shape == (b, h, T, T), f'expected ({b},{h},{T},{T}), got {out.shape}'
    for bi in range(b):
        for hi in range(h):
            assert t.equal(out[bi, hi], mask_2d), f'(b={bi},h={hi}) slice differs from mask_2d'
    # -inf locations preserved (no accidental fill).
    assert t.isinf(out).sum().item() == b * h * (T * (T - 1) // 2)

    # Visualize the causal mask: dark = blocked (-inf), bright = allowed (0).
    fig, ax = plt.subplots(figsize=(3.5, 3.5))
    viz = mask_2d.clone()
    viz[t.isinf(viz)] = -1.0  # remap -inf so imshow can render it
    im = ax.imshow(viz.numpy(), cmap='viridis')
    ax.set_title(f'Causal mask (T={T})')
    ax.set_xlabel('key position')
    ax.set_ylabel('query position')
    plt.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex6')
    print("ex6 ✓")

_test_ex6()

<details><summary>Solution</summary>

```python
def ex6_broadcast_causal_mask(mask_2d: Tensor, b: int, h: int) -> Tensor:
    return repeat(mask_2d, 'q k -> b h q k', b=b, h=h)
```

**Reading the pattern.** Two new named axes (`b`, `h`) are bound by kwarg and inserted to the **left** of `q k`. einops broadcasts the original `(T, T)` block identically into every `(b, h)` slot — no data is copied semantically (it's a stride-0 broadcast under the hood for the PyTorch backend), so this is essentially free.

**Why this is Colab-only.** A flashcard can ask "what pattern broadcasts a 2D mask into 4D?" but it can't show you the staircase heatmap that makes the causal structure click. Look at the plot: row `i` has bright cells only at columns `0..i` — that's exactly "query i can attend to keys 0..i".
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex6'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex6',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()